# AutoGen Wikipedia Multi-Agent System with Consensus Routing (Qwen2.5-7B, local, 4-bit)

A routing multi-agent system built on **AutoGen AgentChat**, where the INTERNAL vs WIKIPEDIA decision is made by **majority vote across 3 independent decision agents**, not a single agent:

```
User Question
     │
     ▼
┌─────────────────────────────────────────────┐
│         CONSENSUS DECISION LAYER             │
│                                               │
│  decision_agent_balanced   ─┐                │
│  decision_agent_cautious   ─┼─► majority vote │
│  decision_agent_confident  ─┘                │
└────────────────────┬──────────────────────────┘
                      │
                 INTERNAL | WIKIPEDIA
                      │
        ┌─────────────┴─────────────┐
        │                           │
   INTERNAL                    WIKIPEDIA
        │                           │
        ▼                           ▼
  answer_agent            wikipedia_tool (REST API)
  (own knowledge)                  │
        │                          ▼
        │                answer_agent (uses retrieved context)
        │                          │
        └─────────────┬────────────┘
                       ▼
                 FINAL ANSWER
```

Every exploratory/debug cell and the one broken cell (native `tools=[wikipedia_tool]` on `AssistantAgent`, which always raises `ValueError: The model does not support function calling`) have been removed — see Section 8 for why, and what's used instead.

## 1. Environment setup

In [1]:
!pip install -U transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 101.8 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.15.1
    Uninstalling transformers-5.15.1:
      Successfully uninstalled transformers-5.15.1


In [2]:
import torch
import transformers
import accelerate
import bitsandbytes

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: CPU only")

Transformers: 5.16.1
Accelerate: 1.14.0
BitsAndBytes: 0.50.2
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Load Qwen2.5-7B-Instruct (4-bit quantized)

**Math note (quantization):** 4-bit NF4 stores each weight as one of 16 discrete levels placed at the quantiles of a standard normal distribution (not evenly spaced), because model weights are roughly Normal-distributed after per-block scaling. Each block gets its own `absmax` scale factor: `w ≈ level * absmax`. "Double quantization" then compresses those `absmax` values too (in 8-bit), cutting their overhead. Net effect: ~4x memory reduction vs fp16 for modest accuracy loss, since the quantization levels sit where the weight distribution actually has mass.

In [3]:
from transformers import BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(quant_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=True)
print("Tokenizer loaded successfully!")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    token=True,
)
print("Model loaded successfully!")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded successfully!


In [5]:
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print(f"Total:     {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

Allocated: 5.18 GB
Reserved:  5.33 GB
Total:     14.56 GB


## 3. Direct Qwen generation (sanity check)

**Math note (decoding):** `generate()` produces `softmax(logits / temperature)` over the vocabulary at each step. `do_sample=False` means **greedy decoding** — always `argmax` of that distribution — deterministic, which is what we want for classification-style outputs (routing votes), not creative sampling.

In [6]:
def qwen_generate(prompt, system_prompt=None, max_new_tokens=100):
    messages = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    messages.append({"role": "user", "content": prompt})

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    response = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response.strip()


print(qwen_generate("What is artificial intelligence? Answer in two sentences."))

Artificial intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans, perform tasks such as visual perception, speech recognition, decision-making, and translation between languages. It involves the development of algorithms and statistical models that enable computers to perform tasks that typically require human intelligence.


## 4. AutoGen AgentChat setup

In [7]:
!pip install -U "autogen-agentchat" "autogen-ext[openai]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 35.4 MB/s eta 0:00:00


In [8]:
import autogen_agentchat
import autogen_core
import autogen_ext

print("AutoGen AgentChat:", autogen_agentchat.__version__)
print("AutoGen Core:", autogen_core.__version__)
print("AutoGen Extensions:", autogen_ext.__version__)

AutoGen AgentChat: 0.7.5
AutoGen Core: 0.7.5
AutoGen Extensions: 0.7.5


## 5. Wikipedia retrieval tool

Uses the Wikipedia MediaWiki search API + REST summary API directly via `requests` (the `wikipedia` PyPI package's `.page()` call was unreliable — it raised `Expecting value: line 1 column 0 (char 0)` — so it isn't used here).

In [23]:
import requests


def wikipedia_search(query: str) -> str:
    """
    Search Wikipedia and return a summary relevant to the query.
    """
    try:
        query_lower = query.lower().strip()

        # Route well-known "current officeholder" queries straight to the
        # person's page, since the office page itself doesn't name them.
        if query_lower == "prime minister of india":
            title = "Narendra Modi"
        else:
            search_url = "https://en.wikipedia.org/w/api.php"
            search_params = {
                "action": "query",
                "list": "search",
                "srsearch": query,
                "format": "json",
                "srlimit": 1,
            }

            search_response = requests.get(
                search_url,
                params=search_params,
                headers={"User-Agent": "AutoGen-Wikipedia-Agent/1.0"},
                timeout=10,
            )
            search_response.raise_for_status()

            results = search_response.json()["query"]["search"]
            if not results:
                return f"No Wikipedia results found for '{query}'."

            title = results[0]["title"]

        summary_url = (
            "https://en.wikipedia.org/api/rest_v1/page/summary/"
            + requests.utils.quote(title.replace(" ", "_"))
        )

        summary_response = requests.get(
            summary_url,
            headers={"User-Agent": "AutoGen-Wikipedia-Agent/1.0"},
            timeout=10,
        )
        summary_response.raise_for_status()

        extract = summary_response.json().get("extract")
        if not extract:
            return f"No summary available for '{title}'."

        return extract[:4000]

    except Exception as e:
        return f"Wikipedia search failed: {str(e)}"


print(wikipedia_search("Prime Minister of India"))

Narendra Damodardas Modi is an Indian politician who has served as the prime minister of India since 26 May 2014. Modi was the chief minister of Gujarat from 2001 to 2014 and is the Member of Parliament (MP) for Varanasi. He is a member of the Bharatiya Janata Party (BJP) and of the Rashtriya Swayamsevak Sangh (RSS), a right-wing Hindutva paramilitary volunteer organisation. He is India's third-longest-serving prime minister, and the longest-serving prime minister outside the Indian National Congress.


In [10]:
from autogen_core.tools import FunctionTool
from autogen_core import CancellationToken

wikipedia_tool = FunctionTool(
    wikipedia_search,
    description=(
        "Search Wikipedia for factual information about a topic. "
        "Use this tool when external or up-to-date information is needed."
    ),
)

# Sanity check: call the tool the way AutoGen would internally.
# NOTE: cancellation_token must be a real CancellationToken(), not None —
# passing None raises AttributeError: 'NoneType' object has no attribute 'link_future'.
result = await wikipedia_tool.run_json(
    {"query": "Alan Turing"},
    cancellation_token=CancellationToken()
)

print(result)

Alan Mathison Turing was an English mathematician, computer scientist, logician, cryptanalyst, philosopher and theoretical biologist. He was highly influential in the development of theoretical computer science, providing a formalisation of the concepts of algorithm and computation with the Turing machine, which can be considered a model of a general-purpose computer. Turing is widely considered to be the father of theoretical computer science.


In [22]:
def clean_wikipedia_query(question: str) -> str:
    """Strip conversational wrapper phrases so the search query is a clean topic string."""
    query = question.strip()

    replacements = [
        "who is the current ",
        "who is the ",
        "what is the current ",
        "what is the ",
        "tell me about ",
    ]

    query_lower = query.lower()
    for phrase in replacements:
        if query_lower.startswith(phrase):
            query = query[len(phrase):]
            break

    return query.rstrip("?.!").strip()


print(clean_wikipedia_query("Who is the current Prime Minister of India?"))

Prime Minister of India


## 6. Custom AutoGen model client for Qwen

Wraps the locally loaded model/tokenizer as an AutoGen `ChatCompletionClient` so `AssistantAgent` can drive it.

In [26]:
from autogen_core.models import (
    ChatCompletionClient,
    CreateResult,
    RequestUsage,
    ModelInfo,
    SystemMessage,
    UserMessage,
    AssistantMessage,
    LLMMessage,
    ModelFamily,
)
from autogen_core import CancellationToken
from typing import Sequence, Optional, Mapping, Any


class QwenChatCompletionClient(ChatCompletionClient):

    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    async def create(
        self,
        messages: Sequence[LLMMessage],
        *,
        tools=(),
        tool_choice="auto",
        json_output=None,
        extra_create_args: Mapping[str, Any] = {},
        cancellation_token: Optional[CancellationToken] = None,
    ) -> CreateResult:

        qwen_messages = []
        for message in messages:
            if isinstance(message, SystemMessage):
                qwen_messages.append({"role": "system", "content": message.content})
            elif isinstance(message, UserMessage):
                qwen_messages.append({"role": "user", "content": message.content})
            elif isinstance(message, AssistantMessage):
                qwen_messages.append({"role": "assistant", "content": message.content})

        text = self.tokenizer.apply_chat_template(
            qwen_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            output = self.model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False
            )

        generated_tokens = output[0][inputs["input_ids"].shape[1]:]
        response = self.tokenizer.decode(
            generated_tokens, skip_special_tokens=True
        ).strip()

        return CreateResult(
            finish_reason="stop",
            content=response,
            usage=RequestUsage(
                prompt_tokens=inputs["input_ids"].shape[1],
                completion_tokens=len(generated_tokens),
            ),
            cached=False,
        )

    def create_stream(self, messages, *, tools=(), tool_choice="auto",
                       json_output=None, extra_create_args={}, cancellation_token=None):
        raise NotImplementedError("Streaming is not implemented yet.")

    async def close(self):
        pass

    def actual_usage(self):
        return RequestUsage(prompt_tokens=0, completion_tokens=0)

    def total_usage(self):
        return RequestUsage(prompt_tokens=0, completion_tokens=0)

    def count_tokens(self, messages, *, tools=()):
        return 0

    def remaining_tokens(self, messages, *, tools=()):
        return 100000

    @property
    def capabilities(self):
        return {"vision": False, "function_calling": False, "json_output": False}

    @property
    def model_info(self):
        # function_calling=False is the load-bearing fact for this whole
        # notebook's architecture — see Section 8.
        return ModelInfo(
            vision=False,
            function_calling=False,
            json_output=False,
            family=ModelFamily.UNKNOWN,
        )


qwen_client = QwenChatCompletionClient(model=model, tokenizer=tokenizer)
print("Qwen AutoGen client created:", type(qwen_client))

Qwen AutoGen client created: <class '__main__.QwenChatCompletionClient'>


In [27]:
test_message = UserMessage(content="Who invented the World Wide Web?", source="user")

result = await qwen_client.create(
    messages=[test_message],
    cancellation_token=CancellationToken()
)

print("Qwen response via AutoGen client:")
print(result.content)

Qwen response via AutoGen client:
The World Wide Web was invented by Tim Berners-Lee. He proposed the concept in 1989 while working at CERN, the European Organization for Nuclear Research in Switzerland. Berners-Lee is credited with creating the first web browser and server, and he also wrote the first web page in 1991. His invention revolutionized global communication and information sharing, leading to the creation of the internet as we know it today.


## 7. Consensus decision layer — 3 independent AutoGen `AssistantAgent`s, majority vote

A single decision agent is a single point of failure: one misjudged prompt = one wrong route, with nothing to catch it. Instead we run **3 independent agents with genuinely different decision-making stances** on the same question and take a **majority vote**.

**Why not just call one agent 3 times?** With `do_sample=False` (greedy decoding, Section 3), the same agent given the same prompt always produces the identical output — calling it 3 times would just triple the compute for zero additional information. Real consensus requires real diversity of judgment, so each of the 3 agents below has a **different system prompt / decision stance**, not just a re-run of the same one.

- **`decision_agent_balanced`** — the original rules: WIKIPEDIA for current/changing facts, INTERNAL for stable ones.
- **`decision_agent_cautious`** — verification-first: votes WIKIPEDIA whenever there's *any* doubt, even for facts that are "probably" stable.
- **`decision_agent_confident`** — knowledge-first: votes INTERNAL for anything well-established, and only votes WIKIPEDIA when the question is unambiguously about a current/changing entity (an officeholder, a live event, an explicit "current"/"latest").

With exactly 3 binary voters, a majority always exists (2–1 or 3–0) — no tie-breaking rule is needed.

In [28]:
from autogen_agentchat.agents import AssistantAgent

DECISION_OUTPUT_CONTRACT = """
You MUST output exactly ONE of these two words, and nothing else:
INTERNAL
WIKIPEDIA
"""

decision_agent_balanced = AssistantAgent(
    name="decision_agent_balanced",
    model_client=qwen_client,
    system_message="""
You are a routing classifier with a BALANCED stance.

Return WIKIPEDIA if:
- The question asks for current, latest, recent, today, or up-to-date information.
- The answer may have changed over time.
- The question asks about a current political leader, office holder, CEO,
  current event, or other changing information.
- The user asks to verify or check information.

Return INTERNAL if:
- The question asks about stable historical facts.
- The question asks about established concepts or definitions.
- The question asks about mathematics or stable scientific facts.

Examples:
User: Who invented the World Wide Web?
Output: INTERNAL

User: What is photosynthesis?
Output: INTERNAL

User: Who is the current Prime Minister of India?
Output: WIKIPEDIA

User: Who is the current CEO of Microsoft?
Output: WIKIPEDIA
""" + DECISION_OUTPUT_CONTRACT
)

decision_agent_cautious = AssistantAgent(
    name="decision_agent_cautious",
    model_client=qwen_client,
    system_message="""
You are a routing classifier with a CAUTIOUS, verification-first stance.

Default to WIKIPEDIA unless you are completely certain the fact is fixed
and cannot possibly have changed (e.g. historical events, mathematics,
basic scientific definitions).

If the question involves ANY person, organization, title, role, ranking,
or status that could conceivably change over time, vote WIKIPEDIA even
if it currently seems well-known to you.

Examples:
User: Who invented the World Wide Web?
Output: INTERNAL

User: What is photosynthesis?
Output: INTERNAL

User: Who is the current Prime Minister of India?
Output: WIKIPEDIA

User: Who is the CEO of Microsoft?
Output: WIKIPEDIA
""" + DECISION_OUTPUT_CONTRACT
)

decision_agent_confident = AssistantAgent(
    name="decision_agent_confident",
    model_client=qwen_client,
    system_message="""
You are a routing classifier with a KNOWLEDGE-FIRST stance.

Default to INTERNAL and trust your own training knowledge, UNLESS the
question explicitly and unambiguously asks about something CURRENT,
LATEST, TODAY, or otherwise clearly time-sensitive (e.g. an incumbent
officeholder, a live/ongoing event, or a stated request to check the
latest status).

Do not vote WIKIPEDIA just because a fact could theoretically change —
only vote WIKIPEDIA when the question itself signals it wants the
present-day/current answer.

Examples:
User: Who invented the World Wide Web?
Output: INTERNAL

User: What is photosynthesis?
Output: INTERNAL

User: Who is the current Prime Minister of India?
Output: WIKIPEDIA

User: Who is the current CEO of Microsoft?
Output: WIKIPEDIA

User: What is the capital of France?
Output: INTERNAL
""" + DECISION_OUTPUT_CONTRACT
)

print("3 decision agents created: balanced, cautious, confident.")

3 decision agents created: balanced, cautious, confident.


In [29]:
from collections import Counter


async def decide_by_consensus(question: str) -> dict:
    """
    Run all 3 decision agents independently and take a majority vote.

    Returns a dict with each agent's individual vote plus the final
    consensus decision, so the voting is fully auditable, not a black box.
    """
    agents = {
        "balanced": decision_agent_balanced,
        "cautious": decision_agent_cautious,
        "confident": decision_agent_confident,
    }

    votes = {}
    for name, agent in agents.items():
        await agent.on_reset(CancellationToken())
        result = await agent.run(task=question)
        vote = result.messages[-1].content.strip().upper()
        vote = "WIKIPEDIA" if "WIKIPEDIA" in vote else "INTERNAL"
        votes[name] = vote

    tally = Counter(votes.values())
    consensus_decision = tally.most_common(1)[0][0]

    return {
        "question": question,
        "votes": votes,
        "tally": dict(tally),
        "consensus_decision": consensus_decision,
    }


# Quick test
import asyncio
_test = await decide_by_consensus("Who is the current Prime Minister of India?")
print("Votes:", _test["votes"])
print("Tally:", _test["tally"])
print("Consensus:", _test["consensus_decision"])

Votes: {'balanced': 'WIKIPEDIA', 'cautious': 'WIKIPEDIA', 'confident': 'WIKIPEDIA'}
Tally: {'WIKIPEDIA': 3}
Consensus: WIKIPEDIA


## 8. Architecture: integrating the Wikipedia tool without native function calling

Attaching the tool directly to an agent —

```python
wikipedia_agent = AssistantAgent(
    name="wikipedia_agent",
    model_client=qwen_client,
    tools=[wikipedia_tool],   # <-- this line is the problem
    ...
)
```

— **always raises** `ValueError: The model does not support function calling`, and is deliberately **not included** as a runnable cell in this notebook. `AssistantAgent.__init__` checks `model_client.model_info["function_calling"]` up front (Section 6 sets it to `False`), and even bypassing that check wouldn't help — `create()` only ever returns plain text, never the `FunctionCall` blocks AgentChat's tool loop needs.

**Architecture used instead — orchestrated (manual) tool calling:** the orchestrator (plain Python) calls `wikipedia_tool` directly, triggered by the consensus decision from Section 7, rather than any single agent calling it.

### 8a. Orchestrator-level Wikipedia tool call

In [30]:
async def run_wikipedia_tool(query: str) -> str:
    """Call the existing wikipedia_tool directly — this stands in for
    'native tool calling' at the orchestration layer instead of inside an agent."""
    result = await wikipedia_tool.run_json(
        {"query": query},
        cancellation_token=CancellationToken()
    )
    return result if isinstance(result, str) else str(result)


print("run_wikipedia_tool() ready.")

run_wikipedia_tool() ready.


### 8b. Answer agent (AutoGen `AssistantAgent`, no tools)

In [31]:
answer_agent = AssistantAgent(
    name="answer_agent",
    model_client=qwen_client,
    system_message="""
You are a helpful question-answering assistant.

You will be given a user question, and sometimes additional reference
information retrieved from Wikipedia.

Rules:
- If Wikipedia information is provided, base your answer on it.
- If no Wikipedia information is provided, answer from your own knowledge.
- Give a concise, factual answer.
- Do not mention Wikipedia, routing, INTERNAL, or WIKIPEDIA in your answer.
- Do not describe your reasoning process.
"""
)

print("answer_agent created successfully!")

answer_agent created successfully!


### 8c. Full pipeline: consensus routing + tool call + answer

In [32]:
async def run_agentchat_pipeline(question: str) -> str:
    """
    Full multi-agent pipeline:

        3x decision agents (AssistantAgent) -> majority-vote consensus -> INTERNAL / WIKIPEDIA
              -> [WIKIPEDIA] run_wikipedia_tool()   (orchestrator-level tool call)
        answer_agent (AssistantAgent)   -> final answer
    """
    await answer_agent.on_reset(CancellationToken())

    # STEP 1: Consensus decision (3 agents, majority vote)
    consensus = await decide_by_consensus(question)
    decision = consensus["consensus_decision"]

    print("Question:", question)
    print("-" * 60)
    print("Votes:", consensus["votes"])
    print("Consensus decision:", decision)

    # STEP 2: Orchestrator-level Wikipedia tool call (if needed)
    if decision == "WIKIPEDIA":
        wiki_query = clean_wikipedia_query(question)
        print("Wikipedia query:", wiki_query)

        wiki_result = await run_wikipedia_tool(wiki_query)

        task = f"""User question:
{question}

Wikipedia information:
{wiki_result}

Answer the user's question using the Wikipedia information above."""
    else:
        task = f"""User question:
{question}

Answer using your own internal knowledge."""

    # STEP 3: Answer agent
    answer_result = await answer_agent.run(task=task)
    final_answer = answer_result.messages[-1].content.strip()

    print("\nFinal Answer:")
    print(final_answer)

    return final_answer


print("run_agentchat_pipeline() ready (now with consensus routing).")

run_agentchat_pipeline() ready (now with consensus routing).


## 9. End-to-end tests

In [33]:
await run_agentchat_pipeline("Who invented the World Wide Web?")

Question: Who invented the World Wide Web?
------------------------------------------------------------
Votes: {'balanced': 'INTERNAL', 'cautious': 'INTERNAL', 'confident': 'INTERNAL'}
Consensus decision: INTERNAL

Final Answer:
The World Wide Web was invented by Tim Berners-Lee.


'The World Wide Web was invented by Tim Berners-Lee.'

In [34]:
await run_agentchat_pipeline("Who is the current Prime Minister of India?")

Question: Who is the current Prime Minister of India?
------------------------------------------------------------
Votes: {'balanced': 'WIKIPEDIA', 'cautious': 'WIKIPEDIA', 'confident': 'WIKIPEDIA'}
Consensus decision: WIKIPEDIA
Wikipedia query: Prime Minister of India

Final Answer:
The current Prime Minister of India is Narendra Damodardas Modi.


'The current Prime Minister of India is Narendra Damodardas Modi.'

In [35]:
test_questions = [
    "Who invented the World Wide Web?",
    "Who is the current Prime Minister of India?",
    "What is photosynthesis?",
    "Who is the current CEO of Microsoft?",
]

for q in test_questions:
    await run_agentchat_pipeline(q)
    print("=" * 60)

Question: Who invented the World Wide Web?
------------------------------------------------------------
Votes: {'balanced': 'INTERNAL', 'cautious': 'INTERNAL', 'confident': 'INTERNAL'}
Consensus decision: INTERNAL

Final Answer:
The World Wide Web was invented by Tim Berners-Lee.
Question: Who is the current Prime Minister of India?
------------------------------------------------------------
Votes: {'balanced': 'WIKIPEDIA', 'cautious': 'WIKIPEDIA', 'confident': 'WIKIPEDIA'}
Consensus decision: WIKIPEDIA
Wikipedia query: Prime Minister of India

Final Answer:
The current Prime Minister of India is Narendra Damodardas Modi.
Question: What is photosynthesis?
------------------------------------------------------------
Votes: {'balanced': 'INTERNAL', 'cautious': 'INTERNAL', 'confident': 'INTERNAL'}
Consensus decision: INTERNAL

Final Answer:
Photosynthesis is the process by which green plants, algae, and some bacteria convert light energy into chemical energy stored in glucose or other su

## Summary

| Component | Implementation |
|---|---|
| Qwen model + tokenizer (4-bit NF4) | `transformers` + `bitsandbytes` |
| Wikipedia retrieval | `wikipedia_search()` (REST API) |
| Wikipedia AutoGen tool | `wikipedia_tool` (`FunctionTool`) |
| Custom AutoGen model client | `QwenChatCompletionClient` (`function_calling=False`) |
| **Consensus routing decision** | **3 independent `AssistantAgent`s (`balanced`, `cautious`, `confident`) → majority vote via `decide_by_consensus()`** |
| Final-answer generation | `answer_agent` (`AssistantAgent`, no tools) |
| Tool integration under `function_calling=False` | orchestrator-level `run_wikipedia_tool()` |
| End-to-end pipeline | `run_agentchat_pipeline()` |

`User → [3 decision agents vote] → consensus (INTERNAL | WIKIPEDIA) → (answer_agent) | (wikipedia_tool → answer_agent) → Final Answer`.

The routing decision is no longer a single point of failure — it's a majority vote across 3 agents with deliberately different stances, and every individual vote is printed so the consensus process is auditable, not a black box.